# Exercise 4.10

## Part 1 

Step 1 

In [80]:
# Import libraries
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import scipy

In [82]:
# create path
path='/home/haus/DA_Course_CF/Exercise_4_Instacart_Basket_Analysis'

In [84]:
# Create an output for the folder Visualzations
output_dir = '/home/haus/DA_Course_CF/Exercise_4_Instacart_Basket_Analysis/04_Analysis/Visualizations'

In [86]:
# Importing last data-frame from Exercise 4.09
df_copm=pd.read_pickle(r'/home/haus/DA_Course_CF/Exercise_4_Instacart_Basket_Analysis/02_Data/Prepared_Data/cust_ords_prods_merged.pkl')

Step 2 - Consider any security implications that might exist for this new data. You’ll need to address any PII data in the data before continuing your analysis.

In [89]:
# Check data-frame head
df_copm.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,first_order,product_id,add_to_cart_order,...,frequency_flag,first_name,last_name,gender,state,age,date_joined,n_dependants,marital_status,income
0,2539329,1,prior,1,2,8,NaN,True,196,1,...,Non-frequent customer,Linda,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423
1,2539329,1,prior,1,2,8,NaN,True,14084,2,...,Non-frequent customer,Linda,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423
2,2539329,1,prior,1,2,8,NaN,True,12427,3,...,Non-frequent customer,Linda,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423
3,2539329,1,prior,1,2,8,NaN,True,26088,4,...,Non-frequent customer,Linda,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423
4,2539329,1,prior,1,2,8,NaN,True,26405,5,...,Non-frequent customer,Linda,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423


In [91]:
# Retrieve data-frame column name
df_copm.columns

Index(['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow',
       'order_hour_of_day', 'days_since_prior_order', 'first_order',
       'product_id', 'add_to_cart_order', 'reordered', 'Unnamed: 0_y',
       'product_name', 'aisle_id', 'department_id', 'prices', 'merge_info',
       'price_range_loc', 'busiest_day', 'Busiest_days',
       'busiest_period_of_day', 'max_order', 'loyalty_flag', 'average_spend',
       'spending_flag', 'Customer_frequency', 'frequency_flag', 'first_name',
       'last_name', 'gender', 'state', 'age', 'date_joined', 'n_dependants',
       'marital_status', 'income'],
      dtype='object')

The data-frame cotains two columns "first_name" and "last_name" for all customers making them identifiable. Consequently, becuse of privacy regulation (GDPR) and for securoty risks we need to remove these columns. Customers have their unique id numbers (user_id) so there is no need for personal names when analysing the data.

In [94]:
# Create a data frame that does not contain first and last name
df_copm1 = df_copm.drop(['first_name', 'last_name'],  axis =1)

In [95]:
# Check df_copm1.columns
df_copm1.columns

Index(['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow',
       'order_hour_of_day', 'days_since_prior_order', 'first_order',
       'product_id', 'add_to_cart_order', 'reordered', 'Unnamed: 0_y',
       'product_name', 'aisle_id', 'department_id', 'prices', 'merge_info',
       'price_range_loc', 'busiest_day', 'Busiest_days',
       'busiest_period_of_day', 'max_order', 'loyalty_flag', 'average_spend',
       'spending_flag', 'Customer_frequency', 'frequency_flag', 'gender',
       'state', 'age', 'date_joined', 'n_dependants', 'marital_status',
       'income'],
      dtype='object')

In [96]:
# Check data-frame data type
df_copm1.dtypes

order_id                     int64
user_id                      int64
eval_set                    object
order_number                 int64
order_dow                    int64
order_hour_of_day            int64
days_since_prior_order     float64
first_order                   bool
product_id                   int64
add_to_cart_order            int64
reordered                    int64
Unnamed: 0_y                 int64
product_name                object
aisle_id                     int64
department_id                int64
prices                     float64
merge_info                category
price_range_loc             object
busiest_day                 object
Busiest_days                object
busiest_period_of_day       object
max_order                    int64
loyalty_flag                object
average_spend              float64
spending_flag               object
Customer_frequency         float64
frequency_flag              object
gender                      object
state               

In [100]:
# Convert strings to 'category' if they have few unique values (e.g., 'state', 'gender')
df_copm1['state'] = df_copm1['state'].astype('category')
df_copm1['gender'] = df_copm1['gender'].astype('category')

In [101]:
# re-check data-frame data type
df_copm1.dtypes

order_id                     int64
user_id                      int64
eval_set                    object
order_number                 int64
order_dow                    int64
order_hour_of_day            int64
days_since_prior_order     float64
first_order                   bool
product_id                   int64
add_to_cart_order            int64
reordered                    int64
Unnamed: 0_y                 int64
product_name                object
aisle_id                     int64
department_id                int64
prices                     float64
merge_info                category
price_range_loc             object
busiest_day                 object
Busiest_days                object
busiest_period_of_day       object
max_order                    int64
loyalty_flag                object
average_spend              float64
spending_flag               object
Customer_frequency         float64
frequency_flag              object
gender                    category
state               

Step 3 - The Instacart officers are interested in comparing customer behavior in different geographic areas. Create a regional segmentation of the data. You’ll need to create a “Region” column based on the “State” column from your customers data set.
Use the region information in this Wikipedia article to create your column (you only need to create regions, not divisions).
Determine whether there’s a difference in spending habits between the different U.S. regions. (Hint: You can do this by crossing the variable you just created with the spending flag.)

In [105]:
# Importing the regions CSV file
df_regions=pd.read_csv('/home/haus/DA_Course_CF/Exercise_4_Instacart_Basket_Analysis/02_Data/Original_Data/us census bureau regions and divisions.csv')

In [107]:
# Check df_regions columns
df_regions.columns

Index(['State', 'State Code', 'Region', 'Division'], dtype='object')

In [109]:
# Rename column "State" in data-frame df_regions to match the same column name in data-frame df_copm1
df_regions.rename(columns={'State': 'state'}, inplace=True)

In [111]:
# Check df_regions columns
df_regions.columns

Index(['state', 'State Code', 'Region', 'Division'], dtype='object')

In [113]:
# data type for data-frame df_regions
df_regions.dtypes

state         object
State Code    object
Region        object
Division      object
dtype: object

In [115]:
# Check for duplicates in data-frame df_regions
print(df_regions['state'].duplicated().sum())

0


In [117]:
# Convert strings to 'category' if they have few unique values (e.g., 'state', 'Region')
df_regions['state'] = df_regions['state'].astype('category')
df_regions['Region'] = df_regions['Region'].astype('category')

In [119]:
# Merge data-frame df_copm1 (without names) with regions data-frame using column "state" as key
df_copm1=df_copm1.merge(df_regions, on='state', how='left')

In [120]:
# Check df_copm1.columns
df_copm1.columns

Index(['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow',
       'order_hour_of_day', 'days_since_prior_order', 'first_order',
       'product_id', 'add_to_cart_order', 'reordered', 'Unnamed: 0_y',
       'product_name', 'aisle_id', 'department_id', 'prices', 'merge_info',
       'price_range_loc', 'busiest_day', 'Busiest_days',
       'busiest_period_of_day', 'max_order', 'loyalty_flag', 'average_spend',
       'spending_flag', 'Customer_frequency', 'frequency_flag', 'gender',
       'state', 'age', 'date_joined', 'n_dependants', 'marital_status',
       'income', 'State Code', 'Region', 'Division'],
      dtype='object')

In [125]:
# Checking the data-frame head
df_copm1.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,first_order,product_id,add_to_cart_order,...,gender,state,age,date_joined,n_dependants,marital_status,income,State Code,Region,Division
0,2539329,1,prior,1,2,8,NaN,True,196,1,...,Female,Alabama,31,2/17/2019,3,married,40423,AL,South,East South Central
1,2539329,1,prior,1,2,8,NaN,True,14084,2,...,Female,Alabama,31,2/17/2019,3,married,40423,AL,South,East South Central
2,2539329,1,prior,1,2,8,NaN,True,12427,3,...,Female,Alabama,31,2/17/2019,3,married,40423,AL,South,East South Central
3,2539329,1,prior,1,2,8,NaN,True,26088,4,...,Female,Alabama,31,2/17/2019,3,married,40423,AL,South,East South Central
4,2539329,1,prior,1,2,8,NaN,True,26405,5,...,Female,Alabama,31,2/17/2019,3,married,40423,AL,South,East South Central


In [127]:
# Calculate the mean (average) spend per region:
region_spending = df_copm1.groupby('Region', observed=True)['average_spend'].mean().reset_index()

In [129]:
# Create a Chart for the spending regions
custom_palette = {
    'Northeast': '#4B6CB7', # blue
    'Midwest':   '#FFB347', # orange
    'South':     '#4CAF50', # green
    'West':      '#8E44AD'  # purple 
}

plt.figure(figsize=(10,6))
sns.barplot(
    data=region_spending,
    x='Region',
    y='average_spend',
    hue='Region',
    palette=custom_palette,
    legend=False  # No legend needed since x-axis is region
)
plt.title('Average Spend by U.S. Region')
plt.ylabel('Average Spend ($)')
plt.xlabel('Region')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'average_spend_by_region_custom_colors.png'))
plt.close()

Step 4 - Create an exclusion flag for low-activity customers (customers with less than 5 orders) and exclude them from the data. Make sure you export this sample.

In [131]:
# Create exclusion flag for low-activity customers (less than 5 orders)
# This creates a binary indicator where: 1=Low-activity customer (fewer than 5 orders)/0=High-activity customer (5+ orders)
df_copm1['low_activity_flag'] = (df_copm1['order_number'] < 5).astype(int)

In [133]:
# Export the data-frame with all customer and activity flag (1=Low-activity customer (<5 orders)/0=High-activity customer (>5 orders)) to folder
df_copm1.to_pickle(os.path.join(path, '02_Data', 'Prepared_Data', 'activity_customers_full.pkl'))

In [135]:
# Exclude low-activity customers
df_lac_excluded = df_copm1[df_copm1['low_activity_flag'] == 0].copy()

In [137]:
# Export the data-frame with excluded low-activity customer to folder
df_lac_excluded.to_pickle(os.path.join(path, '02_Data', 'Prepared_Data', 'high_activity_customers_sample.pkl'))  

# Close Part 1.1 